In [0]:
%sql
drop table if exists training.delta_demo.people_delta_demo;

In [0]:
from pyspark.sql import Row
data = [Row(id=1, name='Alice', age=30), Row(id=2, name='Bob', age=25), Row(id=3, name='Charlie', age=35)]
df = spark.createDataFrame(data)

df.write.format('delta').mode('overwrite').saveAsTable('training.delta_demo.people_delta_demo')

delta_df = spark.read.table('training.delta_demo.people_delta_demo')
display(delta_df)

In [0]:
df.write.format('delta').mode('overwrite').save("/Volumes/training/delta_demo/my_volume/mytable")

In [0]:
df2=spark.read.load("/Volumes/training/delta_demo/my_volume/mytable")
display(df2)

In [0]:
%sql
describe history training.delta_demo.people_delta_demo


In [0]:
from pyspark.sql.functions import lit

# Create DataFrame with new rows and add 'country' column
data = [Row(id=7, name='Tom'), Row(id=8, name='Herry'), Row(id=9, name='Jerry')]
df = spark.createDataFrame(data).withColumn("country", lit("USA"))
display(df)

In [0]:
df.write.format('delta').mode('append')\
    .option("mergeSchema", "true") \
    .saveAsTable('training.delta_demo.people_delta_demo')

result_df = spark.read.table('training.delta_demo.people_delta_demo')
display(result_df)

In [0]:
%sql
describe history training.delta_demo.people_delta_demo

In [0]:
%sql
select * from  training.delta_demo.people_delta_demo version as of 3;

In [0]:
%sql
select * from  training.delta_demo.people_delta_demo version as of 2;

In [0]:
%sql
select * from  training.delta_demo.people_delta_demo version as of 1;

In [0]:
%sql
select * from training.delta_demo.people_delta_demo timestamp as of '2026-08-19T00:41:56.000+00:00';

In [0]:
%sql
select * from training.delta_demo.people_delta_demo;

In [0]:
%sql
restore training.delta_demo.people_delta_demo version as of 1;

In [0]:
%sql
describe history training.delta_demo.people_delta_demo

In [0]:
from pyspark.sql.functions import lit

# Prepare new data for merge
data = [Row(id=1, name='Tom', age=28), Row(id=7, name='Herry', age=27), Row(id=8, name='Jerry', age=37)]
df = spark.createDataFrame(data)
display(df)

In [0]:
df.createOrReplaceTempView("updates")

merge_sql = """
MERGE INTO training.delta_demo.people_delta_demo AS target
USING updates AS source
ON target.id = source.id
WHEN MATCHED THEN
  UPDATE SET target.name = source.name, target.age = source.age
WHEN NOT MATCHED THEN
  INSERT (id, name, age) VALUES (source.id, source.name, source.age)
"""

spark.sql(merge_sql)

result_df = spark.read.table('training.delta_demo.people_delta_demo').select("id", "name", "age")
display(result_df)

In [0]:
%sql
describe history training.delta_demo.people_delta_demo